# PN28 — three-child residual lift

## tl;dr

The literal two-rung residual correction was a clear negative result. On 30,000 fresh odd anchors, the PN27 base
hit **9.197%**, while the three-child correction hit **4.580%**. Odd corrections broke parity, and the common `-2`
correction moved multiples of 3 back onto multiples of 3. The 35 example remained `59`, but that successful local
case did not generalise.


## Context & Methods

For each pair `(1,13)`, `(3,11)`, `(5,9)`, compute the declared signed completion imbalance. Average the three,
double its ridge displacement twice, and round once. Add that integer residual to the frozen PN27 base candidate.

### Key assumptions

- Exact divisibility gives child completion 1; otherwise completion is `2w/N`.
- Pair orientation is fixed from the lower to higher label.
- Two upward rungs multiply the signed displacement by four.
- Half cases round away from zero.
- No parity repair or retry is allowed.


In [1]:
import csv
import json
from pathlib import Path

HERE = Path.cwd()
results = json.loads((HERE / 'PN28_THREE_CHILD_RESIDUAL_LIFT_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN28_THREE_CHILD_RESIDUAL_LIFT_VALIDATION.json').read_text(encoding='utf-8'))
with (HERE / 'PN28_THREE_CHILD_RESIDUAL_LIFT_VALIDATED_ROWS.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print('status:', results['status'])
print('validation:', validation['checks_passed'], '/', validation['checks_total'])
assert results['status'] == 'NEGATIVE RESULT'
assert validation['all_checks_passed'] is True
assert len(rows) == 60000


status: NEGATIVE RESULT
validation: 8 / 8


## Data

In [2]:
print(results['population'])
assert results['population']['odd_primary_rows'] == 30000
assert results['population']['even_secondary_rows'] == 30000
assert results['population']['protected_87_bit_anchor_used'] is False


{'all_rows': 60000, 'odd_primary_rows': 30000, 'even_secondary_rows': 30000, 'protected_87_bit_anchor_used': False}


## Results — worked example

In [3]:
print(results['worked_example_35'])
assert results['worked_example_35']['base_candidate'] == 59
assert results['worked_example_35']['integer_adjustment'] == 0
assert results['worked_example_35']['corrected_candidate'] == 59
assert results['worked_example_35']['is_prime'] is True


{'base_candidate': 59, 'integer_adjustment': 0, 'corrected_candidate': 59, 'is_prime': True}


## Results — primary odd-anchor comparison

In [4]:
primary = results['odd_primary']
print(primary)
assert primary['base_hits'] == 2759
assert primary['corrected_hits'] == 1374
assert primary['difference'] < 0
for scale, summary in results['odd_by_scale'].items():
    print(scale, summary['base_hit_rate'], summary['corrected_hit_rate'], summary['difference'])
    assert summary['difference'] < 0


{'n': 30000, 'base_hits': 2759, 'base_hit_rate': 0.09196666666666667, 'corrected_hits': 1374, 'corrected_hit_rate': 0.0458, 'difference': -0.04616666666666667, 'difference_95ci_normal': [-0.0489745719801688, -0.043358761353164535], 'gained_hits': 263, 'lost_hits': 1648, 'both_prime': 1111, 'neither_prime': 26978, 'candidate_changed_rate': 0.4876333333333333, 'corrected_candidate_odd_rate': 0.8132666666666667}
high 0.0779 0.0403 -0.0376
low 0.1161 0.057 -0.0591
middle 0.0819 0.0401 -0.0418


## Results — failure mechanisms

In [5]:
for group in results['odd_group_results']:
    if group['dimension'] == 'integer_adjustment':
        print('k=', group['value'], 'n=', group['n'], 'base=', group['base_hit_rate'],
              'corrected=', group['corrected_hit_rate'])
print('relation-broken control:', results['relation_broken_permutation'])
assert results['relation_broken_permutation']['one_sided_p_pooled'] == 1.0


k= -4 n= 1162 base= 0.0 corrected= 0.0
k= -3 n= 753 base= 0.01859229747675963 corrected= 0.0
k= -2 n= 7746 base= 0.1431706687322489 corrected= 0.03782597469661761
k= -1 n= 3725 base= 0.13395973154362417 corrected= 0.0
k= 0 n= 15371 base= 0.06915620323986728 corrected= 0.06915620323986728
k= 1 n= 1124 base= 0.059608540925266906 corrected= 0.0
k= 2 n= 119 base= 0.058823529411764705 corrected= 0.15126050420168066
relation-broken control: {'adjustments': [-4, -3, -2, -1, 0, 1, 2], 'permutations': 10000, 'seed': 28200, 'observed_corrected_rate': 0.0458, 'relation_broken_mean_rate': 0.06783639666666619, 'relation_broken_sd_rate': 0.001115541948293253, 'one_sided_p_pooled': 1.0, 'observed_by_scale': {'high': 0.0403, 'low': 0.057, 'middle': 0.0401}, 'one_sided_p_by_scale': {'high': 1.0, 'low': 1.0, 'middle': 1.0}}


## Results — even anchors

In [6]:
print(results['even_secondary'])
assert results['even_secondary']['base_hits'] == 0
assert results['even_secondary']['corrected_hits'] == 548


{'n': 30000, 'base_hits': 0, 'base_hit_rate': 0.0, 'corrected_hits': 548, 'corrected_hit_rate': 0.018266666666666667, 'difference': 0.018266666666666667, 'difference_95ci_normal': [0.016751260316512487, 0.019782073016820848], 'gained_hits': 548, 'lost_hits': 0, 'both_prime': 0, 'neither_prime': 29452, 'candidate_changed_rate': 0.4961333333333333, 'corrected_candidate_odd_rate': 0.19116666666666668}


## Takeaways

1. `35 -> 59` remains arithmetically correct under the declared residual rule.
2. The rule does not generalise: it approximately halves one-shot accuracy on fresh odd anchors.
3. Odd residual adjustments turn eligible odd candidates into even composites.
4. The `-2` adjustment reverses the PN27 base's useful escape from divisibility by 3.
5. The three-child vector may remain descriptive, but its signed mean is not a valid integer transport law under
   two simple doublings and nearest-integer collapse.
